<a href="https://colab.research.google.com/github/Kavinmathi-Kalaiyarasu/Agentic-AI-stock-research-portfolio-/blob/main/stock_research_%26_portfolio.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q langgraph langchain-google-genai langchain-community tavily-python yfinance numpy pandas

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.6/81.6 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 34.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 39.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 26.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 263.5/263.5 kB 20.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.49.0, but you have google-auth 2.59.1 which is incompatible.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatibl

In [3]:
import os
import getpass

# Enter your Google Gemini API Key
if "GOOGLE_API_KEY" not in os.environ:
    os.environ["GOOGLE_API_KEY"] = getpass.getpass("Enter your Google Gemini API Key: ")

# Optional: Tavily API Key for live news (press Enter to skip if not using)
if "TAVILY_API_KEY" not in os.environ:
    tav_key = getpass.getpass("Enter Tavily API Key (optional, press Enter to skip): ")
    if tav_key.strip():
        os.environ["TAVILY_API_KEY"] = tav_key

print("API setup complete.")

Enter your Google Gemini API Key: ··········
Enter Tavily API Key (optional, press Enter to skip): ··········
API setup complete.


In [27]:
import os
import json
import re
import numpy as np
import pandas as pd
import yfinance as yf
from datetime import datetime
from typing import TypedDict, List, Dict, Any, Optional

from langchain_core.messages import SystemMessage, HumanMessage
from langchain_google_genai import ChatGoogleGenerativeAI
from langgraph.graph import StateGraph, START, END

# Use the supported active Flash model
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0.1)

# ==========================================
# 1. MCP SERVER TOOLS
# ==========================================
def get_price(ticker: str) -> Dict[str, Any]:
    """Returns current and historical price metrics for a ticker."""
    try:
        t = yf.Ticker(ticker)
        info = t.info
        return {
            "ticker": ticker,
            "current_price": info.get("currentPrice") or info.get("regularMarketPrice"),
            "currency": info.get("currency", "INR"),
            "52_week_high": info.get("fiftyTwoWeekHigh"),
            "52_week_low": info.get("fiftyTwoWeekLow")
        }
    except Exception as e:
        return {"ticker": ticker, "error": str(e)}

def get_financials(ticker: str) -> Dict[str, Any]:
    """Returns key ratios and financial statements."""
    try:
        t = yf.Ticker(ticker)
        info = t.info
        return {
            "ticker": ticker,
            "pe_ratio": info.get("trailingPE"),
            "forward_pe": info.get("forwardPE"),
            "debt_to_equity": info.get("debtToEquity"),
            "profit_margins": info.get("profitMargins"),
            "return_on_equity": info.get("returnOnEquity"),
            "revenue": info.get("totalRevenue")
        }
    except Exception as e:
        return {"ticker": ticker, "error": str(e)}

def calc_portfolio_risk(holdings: Dict[str, float]) -> Dict[str, Any]:
    """Computes volatility, beta, and concentration for a set of holdings."""
    tickers = list(holdings.keys())
    weights = np.array(list(holdings.values()))

    # Normalize weights to sum to 1.0 if entered as percentages
    if sum(weights) > 1.5:
        weights = weights / 100.0

    betas = []
    for sym in tickers:
        try:
            t = yf.Ticker(sym)
            b = t.info.get("beta")
            betas.append(b if b is not None else 1.0)
        except Exception:
            betas.append(1.0)

    weighted_beta = float(np.dot(weights, betas))
    hhi = float(np.sum(weights ** 2))
    concentration = "High" if hhi > 0.4 else "Moderate" if hhi > 0.25 else "Diversified"

    return {
        "portfolio": holdings,
        "weighted_beta": round(weighted_beta, 2),
        "concentration_level": concentration,
        "herfindahl_index": round(hhi, 3),
        "risk_explanation": "Higher sensitivity to market swings" if weighted_beta > 1.1 else "Moderate or low market sensitivity"
    }

# ==========================================
# 2. RAG KNOWLEDGE BASE
# ==========================================
RAG_KNOWLEDGE_BASE = {
    "P/E ratio": "Price-to-Earnings ratio measures current share price relative to per-share earnings. High P/E often reflects high growth expectations or overvaluation; low P/E may reflect value or distress.",
    "Debt to Equity": "Financial leverage indicator comparing total debt against shareholders' equity. Values above 1.5 to 2.0 indicate elevated financial leverage.",
    "Beta": "Measures asset volatility in relation to the overall market (benchmark 1.0). A beta > 1 means higher price volatility than the index.",
    "SEBI Investor Guidance": "Retail investors should avoid tips from unverified social channels. Maintain asset diversification and assess underlying profitability."
}

# ==========================================
# 3. GRAPH STATE DEFINITION
# ==========================================
class ResearchState(TypedDict):
    query: str
    tickers: List[str]
    query_type: str
    portfolio_weights: Optional[Dict[str, float]]
    fundamentals_data: Optional[Dict[str, Any]]
    sentiment_data: Optional[Dict[str, Any]]
    risk_data: Optional[Dict[str, Any]]
    rag_context: Optional[str]
    final_report: Optional[str]

# ==========================================
# 4. AGENTS / NODES
# ==========================================
def supervisor_node(state: ResearchState) -> Dict[str, Any]:
    """Identifies tickers, query type, and plans agent calls."""
    prompt = f"""You are the Supervisor in a financial research multi-agent system.
Analyze the user query: "{state['query']}"

Respond ONLY with a valid raw JSON object (no extra commentary):
{{
  "query_type": "stock_research" | "portfolio_risk" | "conceptual",
  "tickers": ["TICKER.NS"],
  "portfolio_weights": {{"TICKER.NS": 0.4}}
}}
Rule: Indian equities must append the '.NS' suffix (e.g., INFY.NS, TCS.NS, RELIANCE.NS, HDFCBANK.NS, TATAMOTORS.NS).
"""
    res = llm.invoke(prompt)
    raw_content = res.content

    # Clean json markdown tags safely
    clean_json = re.sub(r"```json|```", "", raw_content).strip()
    try:
        parsed = json.loads(clean_json)
    except Exception:
        parsed = {"query_type": "stock_research", "tickers": ["INFY.NS"], "portfolio_weights": {}}

    return {
        "query_type": parsed.get("query_type", "stock_research"),
        "tickers": parsed.get("tickers", []),
        "portfolio_weights": parsed.get("portfolio_weights", {})
    }

def fundamental_agent(state: ResearchState) -> Dict[str, Any]:
    """Pulls financials and interprets them using RAG definitions."""
    results = {}
    for ticker in state.get("tickers", []):
        results[ticker] = {
            "price_data": get_price(ticker),
            "financials": get_financials(ticker)
        }

    rag_matches = [desc for term, desc in RAG_KNOWLEDGE_BASE.items() if term.lower() in state["query"].lower()]
    rag_ctx = "\n".join(rag_matches) if rag_matches else RAG_KNOWLEDGE_BASE["SEBI Investor Guidance"]

    return {"fundamentals_data": results, "rag_context": rag_ctx}

def sentiment_agent(state: ResearchState) -> Dict[str, Any]:
    """Gathers recent company news and classifies sentiment."""
    sentiments = {}
    tavily_key = os.environ.get("TAVILY_API_KEY")

    for ticker in state.get("tickers", []):
        news_items = []
        if tavily_key:
            try:
                from tavily import TavilyClient
                tavily = TavilyClient(api_key=tavily_key)
                search_res = tavily.search(query=f"latest stock news earnings {ticker}", max_results=3)
                news_items = [r["content"] for r in search_res.get("results", [])]
            except Exception:
                pass

        if not news_items:
            try:
                t = yf.Ticker(ticker)
                news_items = [n.get("title", "") for n in t.news[:3]] if t.news else ["No major headlines reported."]
            except Exception:
                news_items = ["No recent news available."]

        prompt = f"""Evaluate news sentiment for {ticker}:
{news_items}

Summarize sentiment as Bullish, Neutral, or Cautious with brief supporting evidence."""
        analysis = llm.invoke(prompt).content
        sentiments[ticker] = analysis

    return {"sentiment_data": sentiments}

def risk_agent(state: ResearchState) -> Dict[str, Any]:
    """Runs portfolio risk calculations and explains metrics."""
    if state.get("portfolio_weights"):
        risk_metrics = calc_portfolio_risk(state["portfolio_weights"])
    else:
        risk_metrics = {}
        for ticker in state.get("tickers", []):
            try:
                t = yf.Ticker(ticker)
                risk_metrics[ticker] = {
                    "beta": t.info.get("beta", "N/A"),
                    "52_week_high": t.info.get("fiftyTwoWeekHigh"),
                    "52_week_low": t.info.get("fiftyTwoWeekLow")
                }
            except Exception as e:
                risk_metrics[ticker] = {"error": str(e)}

    return {"risk_data": risk_metrics}

def conceptual_rag_agent(state: ResearchState) -> Dict[str, Any]:
    """Directly answers educational and conceptual questions using RAG."""
    prompt = f"""You are an educational financial assistant.
Explain the following question using these reference definitions:
{json.dumps(RAG_KNOWLEDGE_BASE)}

User Query: {state['query']}

Explain clearly for retail investors. Do NOT provide buy or sell advice."""
    res = llm.invoke(prompt).content
    return {"final_report": res}

def report_agent(state: ResearchState) -> Dict[str, Any]:
    """Combines findings into a structured research note with guardrails."""
    obs_date = datetime.now().strftime("%Y-%m-%d")

    prompt = f"""Synthesize a structured research note from this data:

Query: {state['query']}
Date of Data: {obs_date}
Fundamentals: {json.dumps(state.get('fundamentals_data'))}
News Sentiment: {json.dumps(state.get('sentiment_data'))}
Risk Profile: {json.dumps(state.get('risk_data'))}
Knowledge Base Definitions: {state.get('rag_context')}

STRUCTURE REQUIRED:
1. Business Overview
2. Key Ratios & Fundamentals
3. News Sentiment Analysis
4. Risk Profile & Market Sensitivity
5. Open Questions for Further Due Diligence
6. Mandatory Disclaimer

STRICT GUARDRAILS:
- NEVER state 'buy', 'sell', or give any price targets.
- Include data observation date: {obs_date}.
- Add disclaimer: 'Not Investment Advice - For Educational and Research Purposes Only'.
"""
    report = llm.invoke(prompt).content
    return {"final_report": report}

# ==========================================
# 5. ORCHESTRATION PIPELINE
# ==========================================
workflow = StateGraph(ResearchState)

workflow.add_node("supervisor", supervisor_node)
workflow.add_node("fundamental", fundamental_agent)
workflow.add_node("sentiment", sentiment_agent)
workflow.add_node("risk", risk_agent)
workflow.add_node("conceptual_rag", conceptual_rag_agent)
workflow.add_node("reporter", report_agent)

workflow.add_edge(START, "supervisor")

def route_flow(state: ResearchState):
    if state["query_type"] == "conceptual":
        return "conceptual_rag"
    return ["fundamental", "sentiment", "risk"]

workflow.add_conditional_edges(
    "supervisor",
    route_flow,
    ["conceptual_rag", "fundamental", "sentiment", "risk"]
)

workflow.add_edge("fundamental", "reporter")
workflow.add_edge("sentiment", "reporter")
workflow.add_edge("risk", "reporter")

workflow.add_edge("conceptual_rag", END)
workflow.add_edge("reporter", END)

app = workflow.compile()
print("LangGraph Agent system recompiled successfully!")

LangGraph Agent system recompiled successfully!


In [8]:
import os
from google import genai

client = genai.Client(api_key=os.environ["GOOGLE_API_KEY"])
print("Supported models for your API key:")
for m in client.models.list():
    if "generateContent" in (m.supported_actions or []):
        print(f" - {m.name}")

Supported models for your API key:
 - models/gemini-2.5-flash
 - models/gemini-2.5-pro
 - models/gemini-2.5-flash-preview-tts
 - models/gemini-2.5-pro-preview-tts
 - models/gemma-4-26b-a4b-it
 - models/gemma-4-31b-it
 - models/gemini-flash-latest
 - models/gemini-flash-lite-latest
 - models/gemini-pro-latest
 - models/gemini-2.5-flash-lite
 - models/gemini-2.5-flash-image
 - models/gemini-3-flash-preview
 - models/gemini-3.1-pro-preview
 - models/gemini-3.1-pro-preview-customtools
 - models/gemini-3.1-flash-lite-preview
 - models/gemini-3.1-flash-lite
 - models/gemini-3-pro-image-preview
 - models/gemini-3-pro-image
 - models/nano-banana-pro-preview
 - models/gemini-3.1-flash-image-preview
 - models/gemini-3.1-flash-image
 - models/gemini-3.1-flash-lite-image
 - models/gemini-3.5-flash
 - models/gemini-3.5-flash-lite
 - models/gemini-omni-flash-preview
 - models/gemini-omni-1.1-flash
 - models/gemini-3.5-transcribe
 - models/gemini-3.6-flash
 - models/gemini-3.7-flash
 - models/gemini-

In [28]:
import os
import json
import re
import time
import random
import numpy as np
import pandas as pd
import yfinance as yf
from datetime import datetime
from typing import TypedDict, List, Dict, Any, Optional
from langchain_google_genai import ChatGoogleGenerativeAI
from langgraph.graph import StateGraph, START, END

# Use the officially recommended replacement model from the API error
llm = ChatGoogleGenerativeAI(
    model="gemini-3.5-flash-lite",
    temperature=0.1
)

def safe_invoke(prompt: str, max_retries: int = 5) -> str:
    """Invokes LLM with exponential backoff and jitter to handle rate limits and capacity spikes."""
    delay = 2.0
    for attempt in range(max_retries):
        try:
            res = llm.invoke(prompt)
            content = res.content
            if isinstance(content, str):
                return content
            if isinstance(content, list):
                parts = []
                for p in content:
                    if isinstance(p, str):
                        parts.append(p)
                    elif isinstance(p, dict) and "text" in p:
                        parts.append(p["text"])
                    elif hasattr(p, "text"):
                        parts.append(str(p.text))
                return "".join(parts)
            return str(content)
        except Exception as e:
            err_msg = str(e)
            if ("503" in err_msg or "UNAVAILABLE" in err_msg or "429" in err_msg or "RESOURCE_EXHAUSTED" in err_msg) and attempt < max_retries - 1:
                sleep_time = delay + random.uniform(0.5, 2.0)
                print(f"[Retry {attempt+1}/{max_retries}] Throttled. Retrying in {sleep_time:.1f}s...")
                time.sleep(sleep_time)
                delay *= 2
            else:
                raise e
    return ""

# ==========================================
# 1. MCP SERVER TOOLS
# ==========================================
def get_price(ticker: str) -> Dict[str, Any]:
    """Returns current and historical price metrics for a ticker."""
    try:
        t = yf.Ticker(ticker)
        info = t.info
        return {
            "ticker": ticker,
            "current_price": info.get("currentPrice") or info.get("regularMarketPrice"),
            "currency": info.get("currency", "INR"),
            "52_week_high": info.get("fiftyTwoWeekHigh"),
            "52_week_low": info.get("fiftyTwoWeekLow")
        }
    except Exception as e:
        return {"ticker": ticker, "error": str(e)}

def get_financials(ticker: str) -> Dict[str, Any]:
    """Returns key ratios and financial statements."""
    try:
        t = yf.Ticker(ticker)
        info = t.info
        return {
            "ticker": ticker,
            "pe_ratio": info.get("trailingPE"),
            "forward_pe": info.get("forwardPE"),
            "debt_to_equity": info.get("debtToEquity"),
            "profit_margins": info.get("profitMargins"),
            "return_on_equity": info.get("returnOnEquity"),
            "revenue": info.get("totalRevenue")
        }
    except Exception as e:
        return {"ticker": ticker, "error": str(e)}

def calc_portfolio_risk(holdings: Dict[str, float]) -> Dict[str, Any]:
    """Computes volatility, beta, and concentration for a set of holdings."""
    tickers = list(holdings.keys())
    weights = np.array(list(holdings.values()))

    if sum(weights) > 1.5:
        weights = weights / 100.0

    betas = []
    for sym in tickers:
        try:
            t = yf.Ticker(sym)
            b = t.info.get("beta")
            betas.append(b if b is not None else 1.0)
        except Exception:
            betas.append(1.0)

    weighted_beta = float(np.dot(weights, betas))
    hhi = float(np.sum(weights ** 2))
    concentration = "High" if hhi > 0.4 else "Moderate" if hhi > 0.25 else "Diversified"

    return {
        "portfolio": holdings,
        "weighted_beta": round(weighted_beta, 2),
        "concentration_level": concentration,
        "herfindahl_index": round(hhi, 3),
        "risk_explanation": "Higher sensitivity to broad market fluctuations" if weighted_beta > 1.1 else "Moderate market sensitivity"
    }

# ==========================================
# 2. RAG KNOWLEDGE BASE
# ==========================================
RAG_KNOWLEDGE_BASE = {
    "P/E ratio": "Price-to-Earnings ratio measures current share price relative to per-share earnings. High P/E often reflects high growth expectations or overvaluation; low P/E may reflect value or distress.",
    "Debt to Equity": "Financial leverage indicator comparing total debt against shareholders' equity. High values imply greater debt burden.",
    "Beta": "Measures asset volatility in relation to the overall market (benchmark 1.0). Beta > 1 indicates higher price volatility than the market index.",
    "SEBI Investor Guidance": "Retail investors should avoid tips from unverified social channels. Maintain asset diversification and assess underlying profitability."
}

# ==========================================
# 3. GRAPH STATE DEFINITION
# ==========================================
class ResearchState(TypedDict):
    query: str
    tickers: List[str]
    query_type: str
    portfolio_weights: Optional[Dict[str, float]]
    fundamentals_data: Optional[Dict[str, Any]]
    sentiment_data: Optional[Dict[str, Any]]
    risk_data: Optional[Dict[str, Any]]
    rag_context: Optional[str]
    final_report: Optional[str]

# ==========================================
# 4. AGENTS / NODES
# ==========================================
def supervisor_node(state: ResearchState) -> Dict[str, Any]:
    """Identifies tickers, query type, and plans agent calls."""
    prompt = f"""You are the Supervisor in a financial research multi-agent system.
Analyze the user query: "{state['query']}"
Respond ONLY with raw JSON:
{{
  "query_type": "stock_research",
  "tickers": ["TICKER.NS"],
  "portfolio_weights": {{"TICKER.NS": 0.4}}
}}
Append '.NS' for Indian stocks (e.g., INFY.NS, TCS.NS, RELIANCE.NS, HDFCBANK.NS, TATAMOTORS.NS).
"""
    content_text = safe_invoke(prompt)
    clean_json = content_text.replace("```json", "").replace("```", "").strip()

    try:
        parsed = json.loads(clean_json)
    except Exception:
        tickers = re.findall(r'[A-Z0-9]+\.NS', content_text) or ["INFY.NS"]
        parsed = {"query_type": "stock_research", "tickers": tickers, "portfolio_weights": {}}

    return {
        "query_type": parsed.get("query_type", "stock_research"),
        "tickers": parsed.get("tickers", []),
        "portfolio_weights": parsed.get("portfolio_weights", {})
    }

def fundamental_agent(state: ResearchState) -> Dict[str, Any]:
    """Pulls financials and interprets them using RAG definitions."""
    results = {}
    for ticker in state.get("tickers", []):
        results[ticker] = {
            "price_data": get_price(ticker),
            "financials": get_financials(ticker)
        }
    rag_matches = [desc for term, desc in RAG_KNOWLEDGE_BASE.items() if term.lower() in state["query"].lower()]
    rag_ctx = "\n".join(rag_matches) if rag_matches else RAG_KNOWLEDGE_BASE["SEBI Investor Guidance"]
    return {"fundamentals_data": results, "rag_context": rag_ctx}

def sentiment_agent(state: ResearchState) -> Dict[str, Any]:
    """Gathers recent company news and classifies sentiment."""
    sentiments = {}
    tavily_key = os.environ.get("TAVILY_API_KEY")
    for ticker in state.get("tickers", []):
        news_items = []
        if tavily_key:
            try:
                from tavily import TavilyClient
                tavily = TavilyClient(api_key=tavily_key)
                search_res = tavily.search(query=f"latest stock news earnings {ticker}", max_results=3)
                news_items = [r["content"] for r in search_res.get("results", [])]
            except Exception:
                pass
        if not news_items:
            try:
                t = yf.Ticker(ticker)
                news_items = [n.get("title", "") for n in t.news[:3]] if t.news else ["No major headlines reported."]
            except Exception:
                news_items = ["No recent news available."]

        prompt = f"""Evaluate news sentiment for {ticker}:
{news_items}
Summarize sentiment as Bullish, Neutral, or Cautious with brief supporting evidence."""
        sentiments[ticker] = safe_invoke(prompt)

    return {"sentiment_data": sentiments}

def risk_agent(state: ResearchState) -> Dict[str, Any]:
    """Runs portfolio risk calculations and explains metrics."""
    if state.get("portfolio_weights"):
        risk_metrics = calc_portfolio_risk(state["portfolio_weights"])
    else:
        risk_metrics = {}
        for ticker in state.get("tickers", []):
            try:
                t = yf.Ticker(ticker)
                risk_metrics[ticker] = {
                    "beta": t.info.get("beta", "N/A"),
                    "52_week_high": t.info.get("fiftyTwoWeekHigh"),
                    "52_week_low": t.info.get("fiftyTwoWeekLow")
                }
            except Exception as e:
                risk_metrics[ticker] = {"error": str(e)}
    return {"risk_data": risk_metrics}

def conceptual_rag_agent(state: ResearchState) -> Dict[str, Any]:
    """Directly answers educational and conceptual questions using RAG."""
    prompt = f"""You are an educational financial assistant.
Explain the following question using these reference definitions:
{json.dumps(RAG_KNOWLEDGE_BASE)}
User Query: {state['query']}
Explain clearly for retail investors. Do NOT provide buy or sell advice."""
    return {"final_report": safe_invoke(prompt)}

def report_agent(state: ResearchState) -> Dict[str, Any]:
    """Combines findings into a structured research note with guardrails."""
    obs_date = datetime.now().strftime("%Y-%m-%d")
    prompt = f"""Synthesize a structured research note from this data:
Query: {state['query']}
Date of Data: {obs_date}
Fundamentals: {json.dumps(state.get('fundamentals_data'))}
News Sentiment: {json.dumps(state.get('sentiment_data'))}
Risk Profile: {json.dumps(state.get('risk_data'))}
Knowledge Base Definitions: {state.get('rag_context')}

STRUCTURE REQUIRED:
1. Business Overview
2. Key Ratios & Fundamentals
3. News Sentiment Analysis
4. Risk Profile & Market Sensitivity
5. Open Questions for Further Due Diligence
6. Mandatory Disclaimer

STRICT GUARDRAILS:
- NEVER state 'buy', 'sell', or give any price targets.
- Include data observation date: {obs_date}.
- Add disclaimer: 'Not Investment Advice - For Educational and Research Purposes Only'.
"""
    return {"final_report": safe_invoke(prompt)}

# ==========================================
# 5. ORCHESTRATION PIPELINE
# ==========================================
workflow = StateGraph(ResearchState)
workflow.add_node("supervisor", supervisor_node)
workflow.add_node("fundamental", fundamental_agent)
workflow.add_node("sentiment", sentiment_agent)
workflow.add_node("risk", risk_agent)
workflow.add_node("conceptual_rag", conceptual_rag_agent)
workflow.add_node("reporter", report_agent)

workflow.add_edge(START, "supervisor")

def route_flow(state: ResearchState):
    if state["query_type"] == "conceptual":
        return "conceptual_rag"
    return ["fundamental", "sentiment", "risk"]

workflow.add_conditional_edges(
    "supervisor",
    route_flow,
    ["conceptual_rag", "fundamental", "sentiment", "risk"]
)
workflow.add_edge("fundamental", "reporter")
workflow.add_edge("sentiment", "reporter")
workflow.add_edge("risk", "reporter")
workflow.add_edge("conceptual_rag", END)
workflow.add_edge("reporter", END)

app = workflow.compile()
print("LangGraph Agent system recompiled successfully!")

LangGraph Agent system recompiled successfully!


In [29]:
def ask_assistant(query: str):
    print("=" * 60)
    print(f"USER QUERY: {query}")
    print("=" * 60)
    output = app.invoke({"query": query})
    print(output["final_report"])

# Test 1: Stock Research
ask_assistant("Give me a research summary of Infosys.")

# Test 2: Portfolio Risk
ask_assistant("My portfolio is 40% TCS, 30% HDFC Bank and 30% Reliance. How risky is it?")

# Test 3: Conceptual Educational Query
ask_assistant("What does a P/E ratio of 35 mean for this company?")

USER QUERY: Give me a research summary of Infosys.


/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


**Research Note: Infosys (INFY.NS)**
*Data Observation Date: 2026-10-03*

---

### 1. Business Overview
Infosys is a global leader in next-generation digital services and consulting, enabling clients across various industries to navigate their digital transformation. As of the data observation date, the company generates a substantial revenue base of approximately INR 20.3 billion, backed by strong profitability metrics and widespread market operations.

### 2. Key Ratios & Fundamentals
* **Current Share Price:** 1,035.00 INR
* **52-Week Range:** 980.40 INR (Low) – 1,728.00 INR (High)
* **Valuation:** Trailing P/E Ratio of ~13.30 and Forward P/E Ratio of ~12.94
* **Capital Structure:** Debt-to-Equity ratio of 9.54, indicating a conservative use of leverage
* **Profitability & Returns:** Profit margins stand at approximately 16.37%, accompanied by a strong Return on Equity (ROE) of 32.0%

### 3. News Sentiment Analysis
The current news sentiment for Infosys is **Neutral (leaning Cautiou

/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_

**Data Observation Date:** October 3, 2026

---

### 1. Business Overview
The analyzed portfolio consists of three large-cap Indian corporations across distinct economic sectors:
* **Tata Consultancy Services (TCS.NS - 40% weight):** A leading global IT services, consulting, and business solutions organization.
* **HDFC Bank (HDFCBANK.NS - 30% weight):** One of India's premier private sector banks, offering a wide range of financial services including retail banking, wholesale banking, and treasury operations.
* **Reliance Industries (RELIANCE.NS - 30% weight):** A massive conglomerate with diversified operations spanning hydrocarbon exploration and production, petroleum refining and marketing, retail, and telecommunications/digital services.

---

### 2. Key Ratios & Fundamentals
*Note: Financial metrics reflect data current as of October 3, 2026.*

* **TCS.NS**
  * Current Price: ₹2,075.0 (52-Week Range: ₹1,976.8 – ₹3,350.0)
  * P/E Ratio: 15.08 (Forward P/E: 12.80)
  * Profit Margin

/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


**Research Note**
**Date of Data:** 2026-10-03

### 1. Business Overview
*No direct company name or business operations data was provided in the source dataset. The following analysis is framed around the general implications of the queried metric.*

### 2. Key Ratios & Fundamentals
* **Price-to-Earnings (P/E) Ratio:** 35
* **Data Context:** Based on the provided knowledge base, the P/E ratio measures a company's current share price relative to its per-share earnings. 
* **Observation:** A P/E ratio of 35 is generally considered elevated. According to standard definitions, a high P/E ratio often reflects strong market expectations for future earnings growth, or it may indicate that the equity is overvalued relative to its current earnings output. Without specific fundamental data (such as net income, revenue growth, or industry sector averages), it is not possible to determine whether this valuation is justified by growth potential or driven by market speculation.

### 3. News Sentimen